# Курсовая работа: Анализ задержек и отмен авиарейсов под влиянием внешних факторов
 
Ноутбук формулирует задачу и показывает подбор пяти источников и подготовку данных. Выбран общий период июль 2025 — июнь 2026: по состоянию на 9 ноября 2026 года база NOAA Storm Events опубликована по июнь 2026 включительно. Перед новым запуском проверьте актуальное покрытие NOAA в официальном каталоге. Старые сохранённые результаты очищены, поскольку они относились к другому периоду; перед сдачей выполните все ячейки и сохраните ноутбук. Структура работы: описание источника → проверка данных → краткий вывод.

Казбеков Муслим ИД24-2

## 2.1. Постановка задачи

**Бизнес-проблема:**
Задержки и отмены усложняют выполнение расписания авиакомпаний и работу аэропортов. Их частота и длительность могут различаться по перевозчикам, аэропортам и времени, а неблагоприятная погода может быть связана с ростом числа сбоев. Для планирования работы важно выявить такие различия на фактических данных.

**Бизнес-цель:**
Выявить, как частота рейсов, задержек и отмен меняется по перевозчикам, аэропортам и временным периодам, а также оценить связь операционных нарушений с погодой в аэропортах отправления.

**Критерии достижения:**
- сравнить число рейсов по месяцам, дням недели и времени суток;
- сравнить частоту отмен и задержек, а также длительность задержек по перевозчикам, аэропортам и месяцам;
- проверить, связаны ли погодные условия в аэропорту отправления с задержками и отменами;
- подготовить объединённую таблицу рейсов, справочника аэропортов, погодных данных и зарегистрированных погодных событий и описать её ограничения.

**Практическая ценность:**
Результаты могут помочь увидеть периоды и аэропорты, где чаще возникают сбои, и учитывать их при планировании расписания и ресурсов. Анализ показывает связи в данных, но сам по себе не доказывает, что погода стала причиной конкретной задержки.

**Объект исследования:**
Внутренние коммерческие авиарейсы США, отражённые в статистике BTS.

**Предмет исследования:**
Временные и пространственные различия в частоте задержек и отмен, а также их связь с характеристиками перевозчика, аэропорта отправления и погодными условиями.

**Границы исследования:**
Используются рейсы, отправляющиеся из десяти аэропортов США с наибольшим числом вылетов за выбранный период. В текущем наборе период охватывает 1 июля 2025 — 30 июня 2026 года. Границы выбраны так, чтобы все источники, включая NOAA Storm Events, покрывали один и тот же полный год. Погода связывается с аэропортом отправления и местной датой рейса. Данные не охватывают мировую авиацию и не содержат число пассажиров на каждом рейсе; число рейсов используется только как показатель транспортной активности.

**Исследовательский вопрос:**
Как частота и длительность задержек и вероятность отмен связаны с перевозчиком, аэропортом отправления, месяцем, временем вылета, погодой и зарегистрированными значимыми погодными событиями рядом с аэропортом?

**Гипотезы:**

H1: число рейсов меняется по месяцам и дням недели.

H2: частота задержек и отмен, а также длительность задержек различаются по перевозчикам, аэропортам, времени суток и месяцу.

H3: неблагоприятные погодные условия и зарегистрированные значимые погодные события связаны с более частыми задержками и отменами. NOAA Storm Events отражает только отобранные значимые события, а не все неблагоприятные погодные условия.

H4: число рейсов меняется по времени суток и дням недели.

H0: в выбранной выборке нет различий по числу рейсов, частоте задержек и отмен и длительности задержек между указанными группами и нет связи этих нарушений с погодой. Эти гипотезы предстоит проверять на следующих этапах; подготовка данных сама по себе их не подтверждает.

## 2.2. Подбор и изучение источников данных

В проекте используются пять открытых источников: данные о рейсах, справочник аэропортов, погодный API, сообщения METAR и свободные текстовые описания событий NOAA.

| Формат | Источник и объект | Категориальные признаки | Числовые признаки | Роль |
|---|---|---|---|---|
| CSV в ZIP | [BTS TranStats](https://transtats.bts.gov/ONTIME/); запись на рейс | перевозчик, Origin/Dest, штат, день недели, временной блок | задержки, расстояние, отмена, причины задержки | исходы рейсов и факторы |
| XML | [FreeAirportDB](https://freeairportdb.com/download); запись на аэропорт | коды, тип, страна, регион, часовой пояс | широта, долгота, высота | справочник аэропортов |
| JSON API | [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api); аэропорт-день | аэропорт, дата, WMO-код и класс погоды | температура, осадки, ветер | погодный фон |
| Текстовые сообщения + измерения | [IEM ASOS/METAR](https://mesonet.agron.iastate.edu/ASOS/); сообщение станции | станция, время, коды явлений, аннотация | температура, точка росы, влажность, ветер, осадки, видимость | дополнительный стандартизированный метеотекст и измерения |
| CSV с неструктурированным текстом | [NOAA Storm Events](https://www.ncei.noaa.gov/access/storm-events-database/); значимое погодное событие | штат, тип события, тип зоны, название округа/зоны | травмы, смерти, интенсивность, широта, долгота | свободное текстовое описание погодного события и фактор для рейсов |

### Общие функции загрузки и подготовки

Следующая ячейка содержит код чтения источников, очистки, преобразования единиц, аннотирования и векторизации METAR и NOAA, пространственной привязки штормовых событий, объединения и пересборки данных. Определения функций сами по себе не печатают таблицы; результаты выполнения показываются в следующих ячейках.

In [1]:
from __future__ import annotations

import calendar
import concurrent.futures
import json
import math
import os
import re
import shutil
import sys
import time
import urllib.parse
import urllib.request
import zipfile
from collections import Counter
from pathlib import Path

cwd = Path.cwd().resolve()
# Запускайте Jupyter из папки проекта или задайте COURSEWORK_ROOT перед запуском.
ROOT = Path(os.environ.get("COURSEWORK_ROOT", str(cwd))).expanduser().resolve()
WORK = ROOT / "work"
DATA = WORK / "data"
RAW = DATA / "raw_sources"
OUT = ROOT / "outputs"
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Подключаем PyArrow для сохранения таблицы в формате Parquet.
sys.path.insert(0, str(WORK / "pydeps"))
import pyarrow as pa
_original_unregister = pa.unregister_extension_type
def _safe_unregister(name: str) -> None:
    try:
        _original_unregister(name)
    except pa.ArrowKeyError:
        pass
pa.unregister_extension_type = _safe_unregister

import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

START_DATE, END_DATE = "2025-07-01", "2026-06-30"
PERIOD = "2025_07_2026_06"
NOAA_RADIUS_KM = 50.0
HEADERS = {"User-Agent": "Coursework aviation data preparation/1.0"}
USECOLS = [
    "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "OriginAirportID", "Origin", "DestAirportID", "Dest", "OriginState", "DestState",
    "DayOfWeek", "DepTimeBlk", "CRSDepTime", "DepDelayMinutes", "ArrDelayMinutes",
    "Cancelled", "CancellationCode", "Diverted", "Distance", "WeatherDelay",
    "CarrierDelay", "NASDelay", "SecurityDelay", "LateAircraftDelay"
]

def download(url: str, target: Path, force: bool = False) -> Path:
    if not force and target.exists() and target.stat().st_size > 0:
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".part")
    req = urllib.request.Request(url, headers=HEADERS)
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=120) as response, tmp.open("wb") as dest:
                shutil.copyfileobj(response, dest)
            if tmp.stat().st_size == 0:
                raise IOError("Downloaded an empty file")
            tmp.replace(target)
            return target
        except Exception:
            if tmp.exists(): tmp.unlink()
            if attempt == 2: raise
            time.sleep(3 * (attempt + 1))

def bts_path(year: int, month: int, refresh: bool = False) -> Path:
    name = f"bts_{year}_{month:02d}.zip"
    target = RAW / name
    if not refresh and target.exists() and target.stat().st_size > 0:
        return target
    if not refresh and (year, month) == (2026, 8):
        existing = DATA / "bts_ontime_2026_08.zip"
        if existing.exists():
            shutil.copy2(existing, target)
            return target
    url = ("https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_"
           f"On_Time_Performance_1987_present_{year}_{month}.zip")
    return download(url, target, force=refresh)

def csv_member(path: Path) -> str:
    with zipfile.ZipFile(path) as archive:
        return next(n for n in archive.namelist() if n.lower().endswith(".csv"))

def count_origins(path: Path) -> Counter:
    counts = Counter()
    with zipfile.ZipFile(path) as archive:
        with archive.open(csv_member(path)) as source:
            for chunk in pd.read_csv(source, usecols=["Origin"], dtype={"Origin": "string"}, chunksize=350_000):
                counts.update(chunk.Origin.dropna().str.upper().value_counts().to_dict())
    return counts

def load_selected_flights(paths: list[Path], top10: list[str]) -> tuple[pd.DataFrame, int, int, dict]:
    frames=[]; total_rows=0; full_null=None
    for path in paths:
        with zipfile.ZipFile(path) as archive:
            with archive.open(csv_member(path)) as source:
                for chunk in pd.read_csv(source, usecols=USECOLS, low_memory=False, chunksize=350_000):
                    total_rows += len(chunk)
                    if full_null is None:
                        full_null = chunk.isna().sum()
                    else:
                        full_null = full_null.add(chunk.isna().sum(), fill_value=0)
                    selected=chunk[chunk.Origin.astype("string").str.upper().isin(top10)].copy()
                    if len(selected): frames.append(selected)
    flights=pd.concat(frames,ignore_index=True)
    return flights,total_rows,len(USECOLS),full_null.to_dict()

def load_airport_xml(top10: list[str], refresh: bool = False) -> tuple[pd.DataFrame,int,int,str]:
    xml_source = DATA / "freeairport_airports.xml"
    download("https://freeairportdb.com/download-file?format=xml",xml_source,force=refresh)
    rows=[]; us_rows=[]; generated=""; root_count=0
    with zipfile.ZipFile(xml_source) as archive:
        member=next(n for n in archive.namelist() if n.lower().endswith("airport.xml"))
        with archive.open(member) as source:
            for event, elem in ET.iterparse(source,events=("start","end")):
                if event=="start" and elem.tag=="airports": generated=elem.attrib.get("generated","")
                elif event=="end" and elem.tag=="airport":
                    root_count += 1
                    row={child.tag:(child.text or "").strip() for child in elem}
                    if row.get("country_code")=="US" and row.get("iata_code"):
                        us_rows.append(row)
                        if row["iata_code"] in top10: rows.append(row)
                    elem.clear()
    dimension=(pd.DataFrame(rows).drop_duplicates("iata_code").set_index("iata_code").reindex(top10).reset_index())
    for col in ["latitude","longitude","elevation"]: dimension[col]=pd.to_numeric(dimension[col],errors="coerce")
    return dimension,len(us_rows),root_count,generated

def fetch_weather(airport_dim: pd.DataFrame, dates: tuple[str,str], refresh: bool = False) -> tuple[pd.DataFrame,list[dict]]:
    start,end=dates
    cache_path=RAW/f"openmeteo_weather_{PERIOD}_top10.json"
    if not refresh and cache_path.exists():
        raw=json.loads(cache_path.read_text(encoding="utf-8"))
        results=[(item["airport_iata"],item["payload"]) for item in raw]
    else:
        def one(row):
            params={"latitude":str(row.latitude),"longitude":str(row.longitude),
                    "daily":"weather_code,temperature_2m_max,precipitation_sum,wind_speed_10m_max",
                    "start_date":start,"end_date":end,"timezone":"auto"}
            url="https://archive-api.open-meteo.com/v1/archive?"+urllib.parse.urlencode(params)
            req=urllib.request.Request(url,headers=HEADERS)
            with urllib.request.urlopen(req,timeout=120) as response: payload=json.loads(response.read().decode("utf-8"))
            return row.iata_code,payload
        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:
            results=list(pool.map(one,[row for row in airport_dim.itertuples(index=False)]))
        raw=[{"airport_iata":code,"payload":payload} for code,payload in results]
    rows=[]
    def wmo_class(value):
        if value is None:return "not_reported"
        code=int(value)
        if code==0:return "clear"
        if code in {1,2,3}:return "cloudy"
        if code in {45,48}:return "fog"
        if 51<=code<=57:return "drizzle"
        if 61<=code<=67:return "rain"
        if 71<=code<=77 or code in {85,86}:return "snow"
        if 80<=code<=82:return "showers"
        if 95<=code<=99:return "thunderstorm"
        return "other"
    for code,item in results:
        daily=item["daily"]
        for i,day in enumerate(daily["time"]):
            weather=daily["weather_code"][i]
            rows.append({"airport_iata":code,"FlightDate":day,"weather_code":weather,
                         "weather_class":wmo_class(weather),"temperature_max_c":daily["temperature_2m_max"][i],
                         "precipitation_sum_mm":daily["precipitation_sum"][i],
                         "wind_speed_max_kmh":daily["wind_speed_10m_max"][i],
                         "weather_timezone":item["timezone"]})
    return pd.DataFrame(rows),raw
def fetch_metar(top10: list[str], dates: tuple[str,str], refresh: bool = False) -> pd.DataFrame:
    start = pd.Timestamp(dates[0])
    end_exclusive = pd.Timestamp(dates[1]) + pd.Timedelta(days=1)
    params=[("station",station) for station in top10]
    params += [("data","tmpf,dwpf,relh,drct,sknt,p01i,vsby,wxcodes,metar"),
               ("year1",str(start.year)),("month1",str(start.month)),("day1",str(start.day)),
               ("year2",str(end_exclusive.year)),("month2",str(end_exclusive.month)),("day2",str(end_exclusive.day)),
               ("tz","Etc/UTC"),("format","onlycomma"),("latlon","yes"),("elev","yes"),
               ("missing","M"),("trace","T"),("direct","no"),("report_type","3"),("report_type","4")]
    url="https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"+urllib.parse.urlencode(params)
    path=RAW/f"iem_metar_{PERIOD}_top10.csv"
    if refresh or not path.exists() or path.stat().st_size==0:
        download(url,path,force=refresh)
    return pd.read_csv(path,dtype={"station":"string","wxcodes":"string","metar":"string","p01i":"string"})

WEATHER_GROUP=re.compile(r"[+-]?(?:VC)?(?:MI|PR|BC|DR|BL|SH|FZ)?(?:TS|DZ|RA|SN|SG|IC|PL|GR|GS|UP|BR|FG|FU|VA|DU|SA|HZ|PY|PO|SQ|FC|SS|DS)")
def annotate_metar(text):
    if pd.isna(text) or not str(text).strip():return "not_reported"
    body=str(text).upper().split(" RMK",1)[0]
    groups=[part for part in body.split() if WEATHER_GROUP.fullmatch(part)]
    if any(g.lstrip("+-").startswith(("TS","VCTS")) or g.lstrip("+-")=="TS" for g in groups):return "thunderstorm"
    if any(g.lstrip("+-").endswith(("FG","BR","HZ")) for g in groups):return "fog_or_haze"
    if any(g.lstrip("+-").endswith(("RA","SN","DZ","PL","GR","GS","SG","IC","UP")) for g in groups):return "precipitation"
    return "no_significant_weather"

def process_metar(metar: pd.DataFrame, airport_dim: pd.DataFrame) -> tuple[pd.DataFrame,np.ndarray,list[str],pd.DataFrame]:
    metar=metar.copy()
    metar["doc_id"]=np.arange(len(metar),dtype=np.int32)
    metar["valid_utc"]=pd.to_datetime(metar.valid,errors="coerce",utc=True)
    metar["precip_trace_flag"]=metar.p01i.astype("string").str.upper().eq("T").astype("int8")
    for col in ["tmpf","dwpf","relh","drct","sknt","p01i","vsby","elevation","lat","lon"]:
        if col in metar: metar[col]=pd.to_numeric(metar[col].replace({"M":np.nan,"T":0.0}),errors="coerce")
    metar["temperature_c"]=(metar.tmpf-32)*5/9
    metar["dewpoint_c"]=(metar.dwpf-32)*5/9
    metar["wind_speed_kmh"]=metar.sknt*1.852
    metar["precipitation_mm"]=metar.p01i*25.4
    metar["visibility_km"]=metar.vsby*1.60934
    zone_map=airport_dim.set_index("iata_code").time_zone.to_dict()
    metar["station"]=metar.station.astype("string").str.upper().str.strip()
    metar["local_date"]=pd.Series(index=metar.index,dtype="string")
    for station,group in metar.groupby("station",sort=False):
        if station not in zone_map: continue
        metar.loc[group.index,"local_date"]=group.valid_utc.dt.tz_convert(zone_map[station]).dt.strftime("%Y-%m-%d").to_numpy()
    metar["metar_weather_class"]=metar.metar.map(annotate_metar)
    metar["poor_visibility_flag"]=(metar.vsby<=3).astype("int8")
    daily=(metar.groupby(["station","local_date"],as_index=False)
           .agg(metar_observations=("metar","count"),metar_temp_c_mean=("temperature_c","mean"),
                metar_dewpoint_c_mean=("dewpoint_c","mean"),metar_wind_kmh_mean=("wind_speed_kmh","mean"),
                metar_wind_kmh_max=("wind_speed_kmh","max"),metar_precip_mm_max=("precipitation_mm","max"),
                metar_visibility_km_min=("visibility_km","min"),metar_poor_visibility_reports=("poor_visibility_flag","sum"),
                metar_trace_precip_reports=("precip_trace_flag","sum")))
    class_counts=(metar.pivot_table(index=["station","local_date"],columns="metar_weather_class",values="metar",aggfunc="count",fill_value=0).reset_index())
    class_counts=class_counts.rename(columns={c:f"metar_{c}_reports" for c in class_counts.columns if c not in {"station","local_date"}})
    daily=daily.merge(class_counts,on=["station","local_date"],how="left",validate="one_to_one")
    stations=set(airport_dim.icao_code.astype(str).str.upper())
    stop={"RMK","AO1","AO2","AOA","AUTO","COR","NOSIG","KT","SM","MPS","METAR","SPECI","Z","TEMPO","BECMG"}|stations
    def tokens(value):
        if pd.isna(value):return []
        return [t for t in re.findall(r"\b[A-Z]{2,}[A-Z0-9]*\b",str(value).upper()) if t not in stop and not re.fullmatch(r"[A-Z]?\d+[A-Z0-9]*",t)]
    tokenized=[tokens(t) for t in metar.metar.fillna("")]
    doc_freq=Counter(token for doc in tokenized for token in set(doc))
    vocab=[token for token,_ in doc_freq.most_common(128)]
    n=max(len(tokenized),1); idf={t:math.log((1+n)/(1+doc_freq[t]))+1 for t in vocab}
    columns={t:i for i,t in enumerate(vocab)}
    matrix=np.zeros((n,len(vocab)),dtype=np.float32)
    for i,doc in enumerate(tokenized):
        counts=Counter(t for t in doc if t in columns); denom=max(len(doc),1)
        for token,count in counts.items():matrix[i,columns[token]]=(count/denom)*idf[token]
        norm=float(np.linalg.norm(matrix[i]))
        if norm:matrix[i]/=norm
    message_map=metar[["doc_id","station","valid","valid_utc","local_date","metar","wxcodes","metar_weather_class","precip_trace_flag"]].copy()
    return daily,matrix,vocab,message_map


NOAA_CSV_INDEX = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"

def latest_noaa_details_file(year: int) -> tuple[str, str]:
    """Return the newest official annual details-file URL and its local filename."""
    request = urllib.request.Request(NOAA_CSV_INDEX, headers=HEADERS)
    with urllib.request.urlopen(request, timeout=60) as response:
        catalog = response.read().decode("utf-8", errors="replace")
    pattern = re.compile(rf"StormEvents_details-ftp_v1\.0_d{year}_c(\d{{8}})\.csv\.gz", re.I)
    created = pattern.findall(catalog)
    if not created:
        raise FileNotFoundError(f"NOAA details CSV for data year {year} was not listed in the official catalog")
    creation_date = max(created)
    filename = f"StormEvents_details-ftp_v1.0_d{year}_c{creation_date}.csv.gz"
    return NOAA_CSV_INDEX + filename, filename

def process_noaa_events(dates: tuple[str, str], airport_dim: pd.DataFrame, refresh: bool = False,
                        radius_km: float = NOAA_RADIUS_KM):
    """Load NOAA free-text narratives, map event points to the nearest selected airport,
    aggregate event-day indicators for the flight table, and build a small TF-IDF matrix."""
    start = pd.Timestamp(dates[0]).normalize()
    end = pd.Timestamp(dates[1]).normalize()
    wanted = {
        "event_id", "episode_id", "state", "event_type", "cz_type", "cz_name",
        "begin_date_time", "begin_lat", "begin_lon", "injuries_direct", "deaths_direct",
        "magnitude", "event_narrative", "episode_narrative"
    }
    frames = []
    for year in range(start.year, end.year + 1):
        local_files = list(RAW.glob(f"StormEvents_details-ftp_v1.0_d{year}_c*.csv.gz"))
        if local_files and not refresh:
            source_path = max(local_files, key=lambda p: re.search(r"_c(\d{8})\.csv\.gz$", p.name).group(1))
        else:
            url, filename = latest_noaa_details_file(year)
            source_path = download(url, RAW / filename, force=refresh)
        header = pd.read_csv(source_path, compression="gzip", nrows=0)
        normalized_columns = {column: str(column).replace(chr(65279), "").strip().lower()
                              for column in header.columns}
        selected_columns = [column for column, normalized in normalized_columns.items()
                            if normalized in wanted]
        if not selected_columns:
            raise ValueError(f"No expected NOAA fields were found in {source_path.name}; headers: {list(header.columns)}")
        frame = pd.read_csv(source_path, compression="gzip", low_memory=False,
                            usecols=selected_columns)
        frame = frame.rename(columns={column: normalized_columns[column] for column in selected_columns})
        frames.append(frame)

    source = pd.concat(frames, ignore_index=True, sort=False)
    required = {"event_id", "event_type", "begin_date_time", "begin_lat", "begin_lon",
                "event_narrative", "episode_narrative"}
    missing = sorted(required - set(source.columns))
    if missing:
        raise ValueError(f"NOAA file is missing required columns: {missing}")
    source["begin_datetime_local"] = pd.to_datetime(source["begin_date_time"], errors="coerce")
    source["event_date"] = source["begin_datetime_local"].dt.strftime("%Y-%m-%d")
    source = source.dropna(subset=["begin_datetime_local"]).copy()
    source = source.loc[source["begin_datetime_local"].dt.normalize().between(start, end)].copy()
    events_in_period = len(source)
    source["begin_lat"] = pd.to_numeric(source["begin_lat"], errors="coerce")
    source["begin_lon"] = pd.to_numeric(source["begin_lon"], errors="coerce")
    source["injuries_direct"] = pd.to_numeric(source.get("injuries_direct"), errors="coerce")
    source["deaths_direct"] = pd.to_numeric(source.get("deaths_direct"), errors="coerce")
    source["magnitude"] = pd.to_numeric(source.get("magnitude"), errors="coerce")
    event_text = source["event_narrative"].fillna("").astype(str).str.strip()
    episode_text = source["episode_narrative"].fillna("").astype(str).str.strip()
    source["event_text"] = event_text.mask(event_text.eq(""), episode_text)
    source["event_annotation"] = source["event_type"].fillna("unknown").astype(str).str.strip()
    source["event_id"] = source["event_id"].astype("string")
    source = source.drop_duplicates("event_id", keep="last")

    with_coordinates = source.dropna(subset=["begin_lat", "begin_lon"]).copy()
    airports = airport_dim[["iata_code", "latitude", "longitude"]].copy()
    airports["latitude"] = pd.to_numeric(airports["latitude"], errors="coerce")
    airports["longitude"] = pd.to_numeric(airports["longitude"], errors="coerce")
    airports = airports.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    if airports.empty:
        raise ValueError("Airport coordinates are required to link NOAA events to flights")
    if not with_coordinates.empty:
        event_lat = np.radians(with_coordinates["begin_lat"].to_numpy(dtype=float))[:, None]
        event_lon = np.radians(with_coordinates["begin_lon"].to_numpy(dtype=float))[:, None]
        airport_lat = np.radians(airports["latitude"].to_numpy(dtype=float))[None, :]
        airport_lon = np.radians(airports["longitude"].to_numpy(dtype=float))[None, :]
        dlat = airport_lat - event_lat
        dlon = airport_lon - event_lon
        haversine = np.sin(dlat / 2) ** 2 + np.cos(event_lat) * np.cos(airport_lat) * np.sin(dlon / 2) ** 2
        distances = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(haversine, 0, 1)))
        nearest = distances.argmin(axis=1)
        with_coordinates["airport_iata"] = airports.iloc[nearest]["iata_code"].to_numpy()
        with_coordinates["nearest_airport_km"] = distances[np.arange(len(nearest)), nearest]
        mapped = with_coordinates.loc[with_coordinates["nearest_airport_km"] <= radius_km].copy()
    else:
        mapped = with_coordinates.copy()
        mapped["airport_iata"] = pd.Series(dtype="string")
        mapped["nearest_airport_km"] = pd.Series(dtype="float64")

    if mapped.empty:
        daily = pd.DataFrame(columns=["airport_iata", "event_date", "noaa_event_count",
                                      "noaa_event_type_count", "noaa_narrative_count",
                                      "noaa_nearest_event_km"])
    else:
        daily = (mapped.groupby(["airport_iata", "event_date"], as_index=False)
                 .agg(noaa_event_count=("event_id", "nunique"),
                      noaa_event_type_count=("event_type", "nunique"),
                      noaa_narrative_count=("event_text", lambda values: int(values.fillna("").astype(str).str.strip().ne("").sum())),
                      noaa_nearest_event_km=("nearest_airport_km", "min")))

    narrative_mask = mapped["event_text"].fillna("").astype(str).str.strip().ne("")
    documents = mapped.loc[narrative_mask, "event_text"].astype(str).tolist()
    mapped["doc_id"] = pd.Series(pd.NA, index=mapped.index, dtype="Int64")
    mapped.loc[narrative_mask, "doc_id"] = np.arange(len(documents), dtype=np.int32)
    stop_words = {"THE", "AND", "FOR", "WITH", "FROM", "THAT", "THIS", "WERE", "WAS", "ARE", "HAVE", "HAD", "HAS", "INTO", "OVER", "UNDER", "NEAR", "AFTER", "BEFORE", "AREA", "AREAS", "DURING", "ALONG", "ACROSS", "EVENT", "EVENTS"}
    tokenized = [[token for token in re.findall(r"[A-Z]{3,}", doc.upper()) if token not in stop_words]
                 for doc in documents]
    if tokenized:
        document_frequency = Counter(token for doc in tokenized for token in set(doc))
        vocabulary = [token for token, _ in document_frequency.most_common(128)]
        columns = {token: i for i, token in enumerate(vocabulary)}
        matrix = np.zeros((len(tokenized), len(vocabulary)), dtype=np.float32)
        document_count = max(len(tokenized), 1)
        inverse_frequency = {token: math.log((1 + document_count) / (1 + document_frequency[token])) + 1
                             for token in vocabulary}
        for i, document in enumerate(tokenized):
            counts = Counter(token for token in document if token in columns)
            denominator = max(len(document), 1)
            for token, count in counts.items():
                matrix[i, columns[token]] = (count / denominator) * inverse_frequency[token]
            norm = float(np.linalg.norm(matrix[i]))
            if norm:
                matrix[i] /= norm
    else:
        vocabulary = []
        matrix = np.zeros((0, 0), dtype=np.float32)
    metadata = {
        "events_in_period": int(events_in_period),
        "events_with_coordinates": int(len(with_coordinates)),
        "events_near_airport": int(len(mapped)),
        "narratives": int(len(documents)),
    }
    return mapped, daily, matrix, vocabulary, metadata


def period_months(start: str, end: str) -> tuple[str,str,str,list[tuple[int,int]]]:
    sy,sm=map(int,start.split("-")); ey,em=map(int,end.split("-"))
    if (ey,em)<(sy,sm): raise ValueError("End month must not precede start month")
    months=[]; y,m=sy,sm
    while (y,m)<=(ey,em):
        months.append((y,m))
        y,m=(y+1,1) if m==12 else (y,m+1)
    return f"{sy:04d}-{sm:02d}-01",f"{ey:04d}-{em:02d}-{calendar.monthrange(ey,em)[1]:02d}",f"{sy:04d}_{sm:02d}_{ey:04d}_{em:02d}",months

def build(start: str = "2025-07", end: str = "2026-06", refresh: bool = False) -> dict:
    global START_DATE, END_DATE, PERIOD, MONTHS
    START_DATE,END_DATE,PERIOD,MONTHS=period_months(start,end)
    print("Проверка месячных файлов BTS:",PERIOD,flush=True)
    paths=[]
    for i,ym in enumerate(MONTHS,1):
        path=bts_path(*ym,refresh=refresh)
        paths.append(path)
        print(f"BTS archive {i}/{len(MONTHS)} ready: {path.name}",flush=True)
    origin_counts=Counter()
    for path in paths:origin_counts.update(count_origins(path))
    top10=[code for code,_ in origin_counts.most_common(10)]
    print("Десять аэропортов с наибольшим числом вылетов:",top10)
    flights,full_rows,selected_column_count,full_null_counts=load_selected_flights(paths,top10)
    print("Всего строк BTS",full_rows,"; в выборку отобрано",len(flights))
    flights["FlightDate"]=pd.to_datetime(flights.FlightDate,errors="coerce").dt.strftime("%Y-%m-%d")
    for col in ["Reporting_Airline","Origin","Dest","OriginState","DestState","DepTimeBlk","CancellationCode"]:
        flights[col]=flights[col].astype("string").str.strip().str.upper()
    for col in ["DepDelayMinutes","ArrDelayMinutes","Cancelled","Diverted","Distance","WeatherDelay","CarrierDelay","NASDelay","SecurityDelay","LateAircraftDelay","CRSDepTime","DayOfWeek"]:
        flights[col]=pd.to_numeric(flights[col],errors="coerce")
    flights["Cancelled"]=flights.Cancelled.astype("Int8")
    flights["Diverted"]=flights.Diverted.astype("Int8")
    flights["DepartureHourLocal"]=(flights.CRSDepTime//100).astype("Int64")
    print("\n[1/6 — BTS] Очищенные рейсы:",len(flights),"; аэропорты:",flights.Origin.nunique())
    print(flights[["FlightDate","Reporting_Airline","Origin","Dest","DepDelayMinutes","Cancelled"]].head(3).to_string(index=False))
    airport_dim,us_airports,xml_count,xml_generated=load_airport_xml(top10,refresh=refresh)
    airport_dim.to_csv(OUT/f"airport_dimension_{PERIOD}_top10.csv",index=False,encoding="utf-8-sig")
    print("\n[2/6 — XML] Записей:",xml_count,"; аэропортов США с IATA:",us_airports,"; совпадений:",int(airport_dim.name.notna().sum()))
    print(airport_dim[["iata_code","name","latitude","longitude","time_zone"]].head(3).to_string(index=False))
    weather,weather_raw=fetch_weather(airport_dim,(START_DATE,END_DATE),refresh=refresh)
    (RAW/f"openmeteo_weather_{PERIOD}_top10.json").write_text(json.dumps(weather_raw,ensure_ascii=False),encoding="utf-8")
    print("\n[3/6 — Open-Meteo] Строк аэропорт-день:",len(weather),"; пропуски:",weather.isna().sum().to_dict())
    print(weather.head(3).to_string(index=False))
    metar_raw=fetch_metar(top10,(START_DATE,END_DATE),refresh=refresh)
    metar_path=OUT/f"metar_messages_{PERIOD}_top10.csv"
    metar_clean_for_missing=metar_raw.copy()
    for c in ["tmpf","dwpf","relh","drct","sknt","p01i","vsby"]:
        metar_clean_for_missing[c]=pd.to_numeric(metar_clean_for_missing[c].replace({"M":np.nan,"T":0.0}),errors="coerce")
    metar_daily,tfidf,vocab,message_map=process_metar(metar_raw,airport_dim)
    print("\n[4/6 — METAR] Сообщений:",len(metar_raw),"; станций:",metar_raw.station.nunique(),"; классы:",metar_raw.metar.map(annotate_metar).value_counts().to_dict())
    print("Пропуски после обработки M, %:",metar_clean_for_missing[["tmpf","dwpf","relh","drct","sknt","p01i","vsby"]].isna().mean().mul(100).round(2).to_dict())
    print("TF-IDF:",tfidf.shape)
    print(message_map[["station","valid_utc","local_date","metar_weather_class"]].head(3).to_string(index=False))
    message_map.to_csv(metar_path,index=False,encoding="utf-8-sig")
    tfidf_path=OUT/f"metar_tfidf_{PERIOD}_top10.npz"
    np.savez_compressed(tfidf_path,tfidf=tfidf,vocabulary=np.asarray(vocab),doc_id=np.arange(len(message_map),dtype=np.int32))
    noaa_events,noaa_daily,noaa_tfidf,noaa_vocab,noaa_meta=process_noaa_events((START_DATE,END_DATE),airport_dim,refresh=refresh)
    noaa_event_path=OUT/f"noaa_events_{PERIOD}_top10.csv"
    noaa_daily_path=OUT/f"noaa_daily_{PERIOD}_top10.csv"
    noaa_tfidf_path=OUT/f"noaa_tfidf_{PERIOD}_top10.npz"
    noaa_events.to_csv(noaa_event_path,index=False,encoding="utf-8-sig")
    noaa_daily.to_csv(noaa_daily_path,index=False,encoding="utf-8-sig")
    np.savez_compressed(noaa_tfidf_path,tfidf=noaa_tfidf,vocabulary=np.asarray(noaa_vocab),doc_id=noaa_events.loc[noaa_events.doc_id.notna(),"doc_id"].to_numpy(dtype=np.int32))
    print("\n[5/6 — NOAA Storm Events] Событий за период:",noaa_meta["events_in_period"],"; с координатами:",noaa_meta["events_with_coordinates"],"; в радиусе аэропортов:",noaa_meta["events_near_airport"],"; текстовых описаний:",noaa_meta["narratives"])
    print("Радиус связи:",NOAA_RADIUS_KM,"км; событийных дней:",len(noaa_daily),"; TF-IDF:",noaa_tfidf.shape)
    if len(noaa_events): print(noaa_events[["event_date","airport_iata","event_type","nearest_airport_km","event_text"]].head(3).to_string(index=False))
    airport_join=airport_dim.rename(columns={"iata_code":"Origin","identifier":"airport_identifier","icao_code":"airport_icao",
        "name":"airport_name","type":"airport_type","latitude":"airport_latitude","longitude":"airport_longitude",
        "elevation":"airport_elevation_m","country_code":"airport_country","region_code":"airport_region",
        "municipality":"airport_municipality","time_zone":"airport_timezone"})
    weather_daily=weather.merge(metar_daily,left_on=["airport_iata","FlightDate"],right_on=["station","local_date"],how="left",validate="one_to_one").drop(columns=["station","local_date"])
    prepared=flights.merge(airport_join,on="Origin",how="left",validate="many_to_one")
    prepared=prepared.merge(weather_daily,left_on=["Origin","FlightDate"],right_on=["airport_iata","FlightDate"],how="left",validate="many_to_one").drop(columns=["airport_iata"])
    prepared=prepared.merge(noaa_daily,left_on=["Origin","FlightDate"],right_on=["airport_iata","event_date"],how="left",validate="many_to_one").drop(columns=["airport_iata","event_date"])
    for column in ["noaa_event_count","noaa_event_type_count","noaa_narrative_count"]:
        prepared[column]=prepared[column].fillna(0).astype("int32")
    key=["FlightDate","Reporting_Airline","Flight_Number_Reporting_Airline","Origin","Dest","CRSDepTime"]
    duplicates=int(prepared.duplicated(key).sum())
    print("\n[6/6 — Объединение] Строк:",len(prepared),"; столбцов:",len(prepared.columns),"; повторов ключа:",duplicates)
    print("Покрытие — аэропорты:",round(100*prepared.airport_name.notna().mean(),2),"%; погода:",round(100*prepared.weather_class.notna().mean(),2),"%; METAR:",round(100*prepared.metar_observations.notna().mean(),2),"%; рейсы с сообщением NOAA:",round(100*prepared.noaa_event_count.gt(0).mean(),2),"%")
    print(prepared[["FlightDate","Reporting_Airline","Origin","Dest","ArrDelayMinutes","weather_class","metar_observations","noaa_event_count"]].head(3).to_string(index=False))
    output=OUT/f"prepared_flights_{PERIOD}_top10.parquet"
    prepared.to_parquet(output,index=False,compression="zstd")
    monthly=(prepared.groupby([pd.to_datetime(prepared.FlightDate).dt.to_period("M").astype(str),"Origin"]).size().rename("flights").reset_index().rename(columns={"FlightDate":"month"}))
    monthly.to_csv(OUT/f"monthly_flight_counts_{PERIOD}_top10.csv",index=False,encoding="utf-8-sig")
    full_missing={col:round(100*count/full_rows,2) for col,count in full_null_counts.items()}
    metar_missing={col:round(100*pd.to_numeric(metar_raw[col].replace({"M":np.nan,"T":0.0}),errors="coerce").isna().mean(),2) for col in ["tmpf","dwpf","relh","drct","sknt","p01i","vsby"]}
    profile={
      "period":{"start":START_DATE,"end":END_DATE,"months":len(MONTHS)},
      "BTS":{"full_rows":full_rows,"selected_fields":selected_column_count,"origin_airports":len(origin_counts),
             "carriers":int(flights.Reporting_Airline.nunique()),
             "top10":top10,"pilot_rows":int(len(prepared)),"columns":int(len(prepared.columns)),"duplicate_keys":duplicates,
             "full_missing_pct":full_missing},
      "XML":{"records":xml_count,"generated":xml_generated,"us_airports_with_iata":us_airports,
             "top10_matched":int(airport_dim.airport_name.notna().sum()) if "airport_name" in airport_dim else int(airport_dim.name.notna().sum())},
      "OpenMeteo":{"rows":len(weather),"missing_pct":{c:round(100*weather[c].isna().mean(),2) for c in ["weather_code","temperature_max_c","precipitation_sum_mm","wind_speed_max_kmh"]}},
      "IEM":{"observations":len(metar_raw),"stations":int(metar_raw.station.nunique()),"missing_pct":metar_missing,
             "trace_reports":int(message_map.precip_trace_flag.sum()),"annotations":metar_raw.metar.map(annotate_metar).value_counts().to_dict()},
      "joins":{"airport_pct":round(100*prepared.airport_name.notna().mean(),2),
               "weather_pct":round(100*prepared.weather_class.notna().mean(),2),
               "metar_daily_pct":round(100*prepared.metar_observations.notna().mean(),2),
                "noaa_event_day_pct":round(100*prepared.noaa_event_count.gt(0).mean(),2)},
      "tfidf":{"shape":list(tfidf.shape),"terms":len(vocab),"map_file":metar_path.name},
      "NOAA":{"events_in_period":noaa_meta["events_in_period"],"events_with_coordinates":noaa_meta["events_with_coordinates"],
              "events_near_airport":noaa_meta["events_near_airport"],"narratives":noaa_meta["narratives"],
              "airports":int(noaa_events.airport_iata.nunique()) if len(noaa_events) else 0,
              "radius_km":NOAA_RADIUS_KM,"daily_rows":len(noaa_daily),
              "missing_narrative_pct":round(100*noaa_events.event_text.fillna("").astype(str).str.strip().eq("").mean(),2) if len(noaa_events) else 0.0,
              "event_types":noaa_events.event_type.fillna("unknown").value_counts().to_dict() if len(noaa_events) else {},
              "tfidf_shape":list(noaa_tfidf.shape),"tfidf_terms":len(noaa_vocab),
              "event_file":noaa_event_path.name,"daily_file":noaa_daily_path.name},
      "output_parquet":output.name,"monthly_counts":monthly.to_dict("records"),
      "sample":prepared[["FlightDate","Reporting_Airline","Origin","Dest","ArrDelayMinutes","Cancelled","weather_class","temperature_max_c","metar_observations","noaa_event_count","noaa_event_type_count","metar_thunderstorm_reports","metar_trace_precip_reports"]].head(5).where(pd.notna(prepared[["FlightDate","Reporting_Airline","Origin","Dest","ArrDelayMinutes","Cancelled","weather_class","temperature_max_c","metar_observations","noaa_event_count","noaa_event_type_count","metar_thunderstorm_reports","metar_trace_precip_reports"]]),None).to_dict("records")
    }
    profile_path=OUT/f"profile_prepared_{PERIOD}.json"
    profile_path.write_text(json.dumps(profile,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
    print(json.dumps({k:v for k,v in profile.items() if k not in {"sample","monthly_counts","BTS"}},ensure_ascii=False,indent=2,default=str))
    print("BTS — всего:",profile["BTS"]["full_rows"],"; в выборке:",len(prepared),"; полей:",len(prepared.columns),"; повторов ключа:",duplicates)
    print("Размеры файлов — Parquet:",output.stat().st_size,"байт; TF-IDF:",tfidf_path.stat().st_size,"байт; CSV сообщений:",metar_path.stat().st_size,"байт")
    return profile

print("Библиотеки загружены; период можно задать ниже.")

Библиотеки загружены; период можно задать ниже.


### Сбор данных за выбранный период

Указан интервал июль 2025 — июнь 2026. Это выбранный период исследования, не обязательный минимум из методички. Период согласован с доступным покрытием NOAA на дату подготовки ноутбука. При изменении периода пересчитываются BTS, погода, METAR, NOAA и итоговая таблица.

In [2]:
START_MONTH = "2025-07"
END_MONTH = "2026-06"
REBUILD_DATA = False  # True пересобирает выходные таблицы из сохранённых исходников.
REFRESH_SOURCES = False  # True повторно скачивает исходники, включая актуальный каталог NOAA.
START_DATE, END_DATE, PERIOD, MONTHS = period_months(START_MONTH, END_MONTH)
profile_path = OUT / f"profile_prepared_{PERIOD}.json"
required_files = [profile_path, OUT / f"prepared_flights_{PERIOD}_top10.parquet", OUT / f"airport_dimension_{PERIOD}_top10.csv", OUT / f"metar_messages_{PERIOD}_top10.csv", OUT / f"metar_tfidf_{PERIOD}_top10.npz", OUT / f"monthly_flight_counts_{PERIOD}_top10.csv", RAW / f"openmeteo_weather_{PERIOD}_top10.json", OUT / f"noaa_events_{PERIOD}_top10.csv", OUT / f"noaa_daily_{PERIOD}_top10.csv", OUT / f"noaa_tfidf_{PERIOD}_top10.npz"]
if REBUILD_DATA or not all(p.exists() for p in required_files):
    profile = build(start=START_MONTH, end=END_MONTH, refresh=REFRESH_SOURCES)
    print("Подготовка завершена заново.")
else:
    profile = json.loads(profile_path.read_text(encoding="utf-8"))
    print("Используются уже подготовленные результаты.")
print("Период:", profile["period"]["start"], "—", profile["period"]["end"])

Проверка месячных файлов BTS: 2025_07_2026_06
BTS archive 1/12 ready: bts_2025_07.zip
BTS archive 2/12 ready: bts_2025_08.zip
BTS archive 3/12 ready: bts_2025_09.zip
BTS archive 4/12 ready: bts_2025_10.zip
BTS archive 5/12 ready: bts_2025_11.zip
BTS archive 6/12 ready: bts_2025_12.zip
BTS archive 7/12 ready: bts_2026_01.zip
BTS archive 8/12 ready: bts_2026_02.zip
BTS archive 9/12 ready: bts_2026_03.zip
BTS archive 10/12 ready: bts_2026_04.zip
BTS archive 11/12 ready: bts_2026_05.zip
BTS archive 12/12 ready: bts_2026_06.zip
Десять аэропортов с наибольшим числом вылетов: ['ORD', 'DEN', 'ATL', 'DFW', 'PHX', 'LAX', 'CLT', 'LAS', 'SEA', 'MCO']
Всего строк BTS 7043316 ; в выборку отобрано 2375749

[1/6 — BTS] Очищенные рейсы: 2375749 ; аэропорты: 10
FlightDate Reporting_Airline Origin Dest  DepDelayMinutes  Cancelled
2025-07-01                AA    LAX  DFW              0.0          0
2025-07-04                AA    LAX  DFW              0.0          0
2025-07-05                AA    LAX  DF

C:\Users\User\AppData\Local\Temp\ipykernel_2120\614686462.py:303: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  source["begin_datetime_local"] = pd.to_datetime(source["begin_date_time"], errors="coerce")



[5/6 — NOAA Storm Events] Событий за период: 64921 ; с координатами: 36107 ; в радиусе аэропортов: 1252 ; текстовых описаний: 1252
Радиус связи: 50.0 км; событийных дней: 149 ; TF-IDF: (1252, 128)
event_date airport_iata event_type  nearest_airport_km                                                                                                                                                                                                                                                                          event_text
2025-08-10          DEN       Hail           40.558890                                                                                                                                                                                                                                                          The report was from mPING.
2025-08-10          DEN       Hail           47.781136 Strong to severe thunderstorms develop across northeast Colorado. Large hail was the 

### Источник 1. Рейсы BTS

BTS содержит запись на каждый рейс и поля о перевозчике, маршруте, задержке, отмене и перенаправлении. Для пилотного объединения выбраны десять аэропортов с наибольшим числом вылетов за указанный период.

In [3]:
prepared = pd.read_parquet(OUT / profile["output_parquet"])
print("Строк BTS за период:", profile["BTS"]["full_rows"])
print("Рейсов из выбранных аэропортов:", profile["BTS"]["pilot_rows"])
print("Аэропорты:", ", ".join(profile["BTS"]["top10"]))
print("Дубликаты ключа рейса:", profile["BTS"]["duplicate_keys"])
display(prepared[["FlightDate","Reporting_Airline","Origin","Dest","DepDelayMinutes","ArrDelayMinutes","Cancelled"]].head(5))

Строк BTS за период: 7043316
Рейсов из выбранных аэропортов: 2375749
Аэропорты: ORD, DEN, ATL, DFW, PHX, LAX, CLT, LAS, SEA, MCO
Дубликаты ключа рейса: 0


,FlightDate,Reporting_Airline,Origin,Dest,DepDelayMinutes,ArrDelayMinutes,Cancelled
0,2025-07-01,AA,LAX,DFW,0.0,0.0,0
1,2025-07-04,AA,LAX,DFW,0.0,0.0,0
2,2025-07-05,AA,LAX,DFW,0.0,0.0,0
3,2025-07-07,AA,LAX,DFW,25.0,0.0,0
4,2025-07-08,AA,LAX,DFW,34.0,18.0,0


Результат: в итоговую выборку вошли рейсы из десяти аэропортов. Нулевое число дубликатов означает, что ключ рейса не повторяется. Пропуски в задержках и причинах отмен рассматриваются отдельно: для части рейсов эти поля неприменимы.

### Источник 2. XML-справочник аэропортов

XML связывается с рейсами по коду аэропорта отправления. Координаты и часовой пояс нужны для получения погоды и перевода времени METAR в местную дату.

In [4]:
airport_dim = pd.read_csv(OUT / f"airport_dimension_{PERIOD}_top10.csv")
print("Аэропортов в пилотном справочнике:", len(airport_dim))
print("Совпало с BTS:", profile["XML"]["top10_matched"], "из 10")
display(airport_dim[["iata_code","name","latitude","longitude","time_zone"]].head(10))

Аэропортов в пилотном справочнике: 10
Совпало с BTS: 10 из 10


,iata_code,name,latitude,longitude,time_zone
0,ORD,Chicago O'Hare International Airport,41.978600,-87.904800,America/Chicago
1,DEN,Denver International Airport,39.860027,-104.673792,America/Denver
2,ATL,Hartsfield Jackson Atlanta International Airport,33.636700,-84.428101,America/New_York
3,DFW,Dallas Fort Worth International Airport,32.896801,-97.038002,America/Chicago
4,PHX,Phoenix Sky Harbor International Airport,33.435302,-112.005905,America/Phoenix
5,LAX,Los Angeles International Airport,33.942501,-118.407997,America/Los_Angeles
6,CLT,Charlotte Douglas International Airport,35.214001,-80.943100,America/New_York
7,LAS,Harry Reid International Airport,36.083361,-115.151817,America/Los_Angeles
8,SEA,Seattle–Tacoma International Airport,47.447943,-122.310276,America/Los_Angeles
9,MCO,Orlando International Airport,28.429399,-81.308998,America/New_York


Все десять кодов аэропортов из пилотной выборки найдены в XML. Таблица аэропортов имеет одно наблюдение на аэропорт; её поля добавляются к нескольким рейсам одного аэропорта.

### Источник 3. Погодные данные Open-Meteo (JSON API)

Источник содержит суточную погоду по координатам аэропорта. Это данные модельной сетки, а не показания датчика на конкретной взлётно-посадочной полосе.

In [5]:
weather_payloads = json.loads((RAW / f"openmeteo_weather_{PERIOD}_top10.json").read_text(encoding="utf-8"))
weather_preview = []
for item in weather_payloads:
    daily = item["payload"]["daily"]
    for i, day in enumerate(daily["time"][:2]):
        weather_preview.append({"airport": item["airport_iata"], "date": day, "weather_code": daily["weather_code"][i], "temperature_max_c": daily["temperature_2m_max"][i], "precipitation_mm": daily["precipitation_sum"][i], "wind_max_kmh": daily["wind_speed_10m_max"][i]})
print("Всего аэропорт-дней:", profile["OpenMeteo"]["rows"])
print("Пропуски по выбранным погодным полям:", profile["OpenMeteo"]["missing_pct"])
display(pd.DataFrame(weather_preview).head(8))

Всего аэропорт-дней: 3650
Пропуски по выбранным погодным полям: {'weather_code': np.float64(0.0), 'temperature_max_c': np.float64(0.0), 'precipitation_sum_mm': np.float64(0.0), 'wind_speed_max_kmh': np.float64(0.0)}


,airport,date,weather_code,temperature_max_c,precipitation_mm,wind_max_kmh
0,ORD,2025-07-01,3,30.8,0.0,14.9
1,ORD,2025-07-02,51,32.5,0.1,18.8
2,DEN,2025-07-01,3,32.5,0.0,18.9
3,DEN,2025-07-02,2,33.1,0.0,21.0
4,ATL,2025-07-01,53,31.8,3.7,13.8
5,ATL,2025-07-02,51,31.5,0.2,12.4
6,DFW,2025-07-01,3,32.3,0.0,8.1
7,DFW,2025-07-02,51,34.0,0.2,11.0


Для каждого из десяти аэропортов сформирован ряд по датам. Дата вместе с кодом аэропорта образует ключ погодной записи.

### Источник 4. Текстовые сообщения IEM ASOS/METAR

METAR — стандартизированные сообщения о погоде в аэропорту. В таблице сохраняются исходный текст и отдельные измерения. Текст размечается по погодным явлениям и переводится в матрицу TF-IDF. Это дополнительный полуструктурированный источник; требование о свободном неструктурированном тексте покрывается описаниями NOAA ниже.

In [6]:
message_map = pd.read_csv(OUT / profile["tfidf"]["map_file"])
missing_metar = pd.Series(profile["IEM"]["missing_pct"], name="Пропуски после обработки M, %")
print("Сообщений:", profile["IEM"]["observations"], "; станций:", profile["IEM"]["stations"])
print("Размер TF-IDF:", profile["tfidf"]["shape"])
print("Следовые осадки T:", profile["IEM"]["trace_reports"], "сообщений")
display(missing_metar.to_frame())
display(pd.Series(profile["IEM"]["annotations"], name="Сообщений по классу").to_frame())
display(message_map[["doc_id","station","valid_utc","local_date","metar","metar_weather_class"]].head(5))

Сообщений: 100119 ; станций: 10
Размер TF-IDF: [100119, 128]
Следовые осадки T: 5804 сообщений


,"Пропуски после обработки M, %"
tmpf,0.03
dwpf,0.03
relh,0.03
drct,3.45
sknt,0.07
p01i,0.01
vsby,0.02


,Сообщений по классу
no_significant_weather,87267
fog_or_haze,8662
precipitation,3418
thunderstorm,772


,doc_id,station,valid_utc,local_date,metar,metar_weather_class
0,0,ATL,2025-07-01 00:15:00+00:00,2025-06-30,KATL 010015Z 27006KT 10SM TS SCT040CB SCT065 B...,thunderstorm
1,1,ORD,2025-07-01 00:51:00+00:00,2025-06-30,KORD 010051Z 28009KT 10SM FEW090 OVC140 29/16 ...,no_significant_weather
2,2,PHX,2025-07-01 00:51:00+00:00,2025-06-30,KPHX 010051Z 29013G20KT 10SM FEW100 FEW250 46/...,no_significant_weather
3,3,CLT,2025-07-01 00:52:00+00:00,2025-06-30,KCLT 010052Z 19008KT 10SM SCT050 SCT060 SCT080...,no_significant_weather
4,4,ATL,2025-07-01 00:52:00+00:00,2025-06-30,KATL 010052Z 27006G18KT 10SM TS SCT038CB SCT06...,thunderstorm


Аннотации разделяют сообщения на грозу, осадки, туман/дымку и сообщения без значимого явления. Размер матрицы TF-IDF должен совпадать с количеством сообщений по строкам; в этом запуске используется 128 текстовых признаков.

### Источник 5. Свободный текст NOAA Storm Events

Официальная база NOAA/NCEI публикует значимые погодные события в CSV. В полях event_narrative и episode_narrative находятся свободные описания отдельных событий и эпизодов. События связываются с ближайшим выбранным аэропортом по координатам в радиусе 50 км и по дате начала события. Поле event_type служит тематической аннотацией, а текстовые описания переводятся в TF-IDF. Источник охватывает только зарегистрированные значимые явления; отсутствие записи не означает отсутствия неблагоприятной погоды.

Источники: [база NOAA Storm Events](https://www.ncei.noaa.gov/access/storm-events-database/), [каталог CSV](https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/), [описание полей](https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf).

In [7]:
noaa_events = pd.read_csv(OUT / profile["NOAA"]["event_file"])
noaa_daily = pd.read_csv(OUT / profile["NOAA"]["daily_file"])
with np.load(OUT / f"noaa_tfidf_{PERIOD}_top10.npz") as vectors:
    noaa_tfidf_shape = vectors["tfidf"].shape
    noaa_vocabulary_size = len(vectors["vocabulary"])
print("Событий NOAA в периоде:", profile["NOAA"]["events_in_period"])
print("Событий с координатами:", profile["NOAA"]["events_with_coordinates"])
print("Событий в радиусе аэропортов:", profile["NOAA"]["events_near_airport"])
print("Событий с текстом:", profile["NOAA"]["narratives"])
print("Доля рейсов с записью NOAA в радиусе:", profile["joins"]["noaa_event_day_pct"], "%")
print("TF-IDF: строки × признаки =", noaa_tfidf_shape, "; словарь:", noaa_vocabulary_size)
display(noaa_events[["doc_id","event_date","airport_iata","nearest_airport_km","state","event_type","event_annotation","event_text"]].head(5))
display(pd.Series(profile["NOAA"]["event_types"], name="Количество событий по типу").sort_values(ascending=False).head(15).to_frame())

Событий NOAA в периоде: 64921
Событий с координатами: 36107
Событий в радиусе аэропортов: 1252
Событий с текстом: 1252
Доля рейсов с записью NOAA в радиусе: 4.94 %
TF-IDF: строки × признаки = (1252, 128) ; словарь: 128


,doc_id,event_date,airport_iata,nearest_airport_km,state,event_type,event_annotation,event_text
0,0,2025-08-10,DEN,40.558890,COLORADO,Hail,Hail,The report was from mPING.
1,1,2025-08-10,DEN,47.781136,COLORADO,Hail,Hail,Strong to severe thunderstorms develop across ...
2,2,2025-08-10,DEN,47.028427,COLORADO,Hail,Hail,The report was from mPING.
3,3,2025-08-10,DEN,47.484408,COLORADO,Hail,Hail,The report was from mPING.
4,4,2025-08-10,DEN,39.450790,COLORADO,Hail,Hail,The report was from mPING.


,Количество событий по типу
Thunderstorm Wind,647
Hail,272
Flash Flood,180
Flood,64
Marine Thunderstorm Wind,28
Lightning,20
Heavy Rain,16
Tornado,10
Marine High Wind,6
Debris Flow,5


### Сводная пригодность источников

Каждый источник содержит временной признак, категориальные и числовые поля. NOAA даёт свободные описания значимых событий; их координаты связываются с ближайшим выбранным аэропортом в радиусе 50 км, а события агрегируются до аэропорт-дня.

In [8]:
source_summary = pd.DataFrame([
 ["BTS", profile["BTS"]["full_rows"], "рейс", "дата, аэропорт, перевозчик", "задержки, расстояние, отмена"],
 ["XML", profile["XML"]["records"], "аэропорт", "IATA/ICAO, страна, тип, часовой пояс", "широта, долгота, высота"],
 ["Open-Meteo", profile["OpenMeteo"]["rows"], "аэропорт-день", "аэропорт, дата, класс погоды", "температура, осадки, ветер"],
 ["IEM METAR", profile["IEM"]["observations"], "сообщение", "станция, время, коды и класс", "температура, ветер, осадки, видимость"],
 ["NOAA Storm Events", profile["NOAA"]["events_near_airport"], "погодное событие", "штат, тип события, зона, дата", "травмы, смерти, интенсивность, координаты"]
], columns=["Источник","Записей","Единица наблюдения","Категориальные поля","Числовые поля"])
display(source_summary)

,Источник,Записей,Единица наблюдения,Категориальные поля,Числовые поля
0,BTS,7043316,рейс,"дата, аэропорт, перевозчик","задержки, расстояние, отмена"
1,XML,85770,аэропорт,"IATA/ICAO, страна, тип, часовой пояс","широта, долгота, высота"
2,Open-Meteo,3650,аэропорт-день,"аэропорт, дата, класс погоды","температура, осадки, ветер"
3,IEM METAR,100119,сообщение,"станция, время, коды и класс","температура, ветер, осадки, видимость"
4,NOAA Storm Events,1252,погодное событие,"штат, тип события, зона, дата","травмы, смерти, интенсивность, координаты"


## 2.3. Предварительный анализ, очистка и слияние

Далее проверяются типы данных, пропуски, дубликаты, соответствие ключей и примеры объединённых строк. Числа ниже описывают подготовленный набор и сами по себе не доказывают гипотезы.

### 2.3.1. Физический смысл, шкалы и роли полей

| Источник и поля | Физический смысл | Шкала | Роль в анализе |
|---|---|---|---|
| BTS: дата, перевозчик, Origin/Dest, день недели, временной блок | время, участники и маршрут рейса | дата / номинальная / ID | ключи, группировки, факторы H1/H2/H4 |
| BTS: DepDelayMinutes, ArrDelayMinutes и причины задержки | длительность задержки в минутах | отношений | исходы H2; причины применимы не ко всем рейсам |
| BTS: Cancelled, Diverted, CancellationCode | факт отмены/перенаправления и причина | бинарная / номинальная | исходы H2 |
| BTS: Distance, CRSDepTime | расстояние и плановое местное время | отношений / время HHMM | контроль маршрута и временная группировка |
| XML: коды, тип, регион, город, часовой пояс | описание аэропорта | ID / номинальная | соединение по Origin, перевод времени |
| XML: latitude, longitude, elevation | координаты и высота аэропорта | пространственная / отношений | запрос погоды и географический контекст |
| Open-Meteo: weather_code, weather_class, date, timezone | суточное состояние погоды | номинальная / дата | внешний фактор и ключ аэропорт-день |
| Open-Meteo: temperature_max_c, precipitation_sum_mm, wind_speed_max_kmh | температура, осадки и ветер | интервальная / отношений | погодные факторы |
| METAR: station, valid_utc, wxcodes, metar_weather_class | станция, время и кодированные явления | ID / время / номинальная / текст | ключ, аннотация, векторизация |
| METAR: tmpf, dwpf, relh, drct, sknt, p01i, vsby | температура, влажность, ветер, осадки, видимость | интервальная / отношений / круговая | погодные факторы; единицы преобразуются |
| NOAA: event_date, state, event_type, cz_type, cz_name, event_text | время, место, вид погодного события и его свободное описание | дата / номинальная / текст | фактор для H3; описание события аннотируется по event_type и векторизуется TF-IDF |
| NOAA: injuries_direct, deaths_direct, magnitude, begin_lat, begin_lon | последствия события, его измеренная интенсивность и координаты | отношений / пространственная | описательные характеристики события; часть полей имеет пропуски по смыслу |

### 2.3.2. Проверка итоговой таблицы и пропусков

In [9]:
flight_key = ["FlightDate","Reporting_Airline","Flight_Number_Reporting_Airline","Origin","Dest","CRSDepTime"]
print("Форма итоговой таблицы:", prepared.shape)
print("Дубликаты ключа:", int(prepared.duplicated(flight_key).sum()))
print("Покрытие справочника/JSON/METAR и доля рейсов с зарегистрированным событием NOAA, %:", profile["joins"])
print("Пропуски в исходной полной BTS-таблице, %:")
display(pd.Series(profile["BTS"]["full_missing_pct"], name="Пропуски, %").sort_values(ascending=False).to_frame())
print("Примеры объединённых записей:")
display(prepared[["FlightDate","Reporting_Airline","Origin","Dest","ArrDelayMinutes","Cancelled","weather_class","temperature_max_c","metar_observations","noaa_event_count","noaa_nearest_event_km"]].head(5))

Форма итоговой таблицы: (2375749, 58)
Дубликаты ключа: 0
Покрытие справочника/JSON/METAR и доля рейсов с зарегистрированным событием NOAA, %: {'airport_pct': np.float64(100.0), 'weather_pct': np.float64(100.0), 'metar_daily_pct': np.float64(100.0), 'noaa_event_day_pct': np.float64(4.94)}
Пропуски в исходной полной BTS-таблице, %:


,"Пропуски, %"
CancellationCode,98.19
SecurityDelay,78.05
LateAircraftDelay,78.05
NASDelay,78.05
CarrierDelay,78.05
WeatherDelay,78.05
ArrDelayMinutes,2.09
DepDelayMinutes,1.75
OriginState,0.00
OriginAirportID,0.00


Примеры объединённых записей:


,FlightDate,Reporting_Airline,Origin,Dest,ArrDelayMinutes,Cancelled,weather_class,temperature_max_c,metar_observations,noaa_event_count,noaa_nearest_event_km
0,2025-07-01,AA,LAX,DFW,0.0,0,cloudy,21.4,32,0,NaN
1,2025-07-04,AA,LAX,DFW,0.0,0,cloudy,22.8,31,0,NaN
2,2025-07-05,AA,LAX,DFW,0.0,0,cloudy,24.8,29,0,NaN
3,2025-07-07,AA,LAX,DFW,0.0,0,cloudy,21.7,27,0,NaN
4,2025-07-08,AA,LAX,DFW,18.0,0,cloudy,22.0,30,0,NaN


Пустые значения в задержках часто относятся к отменённым или не завершённым рейсам; причины отмены и задержки применимы только к соответствующим случаям. Они не заменялись нулями. Ноль в noaa_event_count означает, что в базе NOAA не найдено соответствующее событие в радиусе 50 км на эту дату; это не доказывает отсутствие плохой погоды. В итоговом наборе ключ рейса уникален, а соединения проверены по аэропорту, дате и близости события.

### 2.3.3. Помесячная динамика рейсов

In [10]:
monthly = pd.DataFrame(profile["monthly_counts"])
monthly_totals = monthly.groupby("month", as_index=False)["flights"].sum()
display(monthly_totals)

,month,flights
0,2025-07,212615
1,2025-08,202755
2,2025-09,191328
3,2025-10,205313
4,2025-11,192422
5,2025-12,195921
6,2026-01,183448
7,2026-02,173387
8,2026-03,205278
9,2026-04,201791


Помесячная таблица показывает объём пилотной выборки в каждом месяце. По ней можно описывать динамику, но для проверки H1 нужно отдельно сравнить месяцы и дни недели статистически.

### 2.3.4. Аннотирование и векторизация текста

In [11]:
with np.load(OUT / f"metar_tfidf_{PERIOD}_top10.npz") as vectors:
    tfidf_shape = vectors["tfidf"].shape
    vocabulary_size = len(vectors["vocabulary"])
print("Сообщения в карте:", len(message_map))
print("TF-IDF: строки × признаки =", tfidf_shape)
print("Размер словаря:", vocabulary_size)
print("Аннотации:")
display(message_map["metar_weather_class"].value_counts().rename("Количество").to_frame())

Сообщения в карте: 100119
TF-IDF: строки × признаки = (100119, 128)
Размер словаря: 128
Аннотации:


,Количество
metar_weather_class,
no_significant_weather,87267
fog_or_haze,8662
precipitation,3418
thunderstorm,772


Каждая строка матрицы соответствует сообщению METAR или событию NOAA по doc_id, столбец — текстовому признаку. METAR стандартизирован; свободный текст представлен полями event_narrative и episode_narrative NOAA. event_type используется как исходная тематическая аннотация, а повествовательный текст переводится в числа TF-IDF.

### 2.3.5. Ключи объединения и хранение

- BTS + XML: код Origin = IATA-код аэропорта; к одному аэропорту относятся многие рейсы.
- BTS + Open-Meteo: Origin + FlightDate = аэропорт-день.
- BTS + METAR: станция + местная дата; почасовые сообщения агрегируются до аэропорт-дня.
- BTS + NOAA: дата события и расстояние от его координат до ближайшего выбранного аэропорта не более 50 км; события агрегируются до аэропорт-дня. Это приближённая привязка для анализа возможного фактора, а не доказательство влияния события на конкретный рейс.

Факты рейсов сохраняются в Parquet со сжатием Zstandard. Матрицы TF-IDF хранятся отдельно в NPZ; CSV-таблицы связывают doc_id с исходными текстами.

### Обновление месячных разделов

Функция ниже записывает подготовленные данные в месячные Parquet-партиции и проверяет ключи и схему. Для полного обновления исходников установите REFRESH_SOURCES=True и вызовите refresh_and_update: она повторно загружает BTS, XML, Open-Meteo, METAR и NOAA, пересобирает слияние, затем обновляет месячные партиции.

In [12]:
from __future__ import annotations

import os
from pathlib import Path
from uuid import uuid4

import pandas as pd

FLIGHT_KEY=["FlightDate","Reporting_Airline","Flight_Number_Reporting_Airline","Origin","Dest","CRSDepTime"]

def write_month(group: pd.DataFrame, target: Path) -> Path:
    if group.empty:
        raise ValueError("Refusing to write an empty month partition")
    missing=sorted(set(FLIGHT_KEY)-set(group.columns))
    if missing:
        raise ValueError(f"Missing flight-key columns: {missing}")
    if group.duplicated(FLIGHT_KEY).any():
        raise ValueError("Duplicate flight keys found; refusing to replace the partition")
    if target.exists():
        old=pd.read_parquet(target)
        old_schema=[(name,str(dtype)) for name,dtype in old.dtypes.items()]
        new_schema=[(name,str(dtype)) for name,dtype in group.dtypes.items()]
        if old_schema!=new_schema:
            raise ValueError(f"Schema/type mismatch for {target}")
    target.parent.mkdir(parents=True,exist_ok=True)
    temp=target.parent/f".{target.stem}-{uuid4().hex}.tmp.parquet"
    try:
        group.to_parquet(temp,index=False,compression="zstd")
        check=pd.read_parquet(temp)
        if len(check)!=len(group) or list(check.columns)!=list(group.columns):
            raise IOError(f"Read-back validation failed for {temp}")
        if check.duplicated(FLIGHT_KEY).any():
            raise IOError(f"Duplicate flight keys in staged partition {temp}")
        os.replace(temp,target)
    finally:
        if temp.exists():temp.unlink()
    return target

def update_partitions(input_path: Path, root: Path) -> list[Path]:
    frame=pd.read_parquet(input_path)
    if frame.empty:
        raise ValueError("Input table is empty")
    missing=sorted(set(FLIGHT_KEY)-set(frame.columns))
    if missing:
        raise ValueError(f"Missing flight-key columns: {missing}")
    dates=pd.to_datetime(frame["FlightDate"],errors="coerce")
    if dates.isna().any():
        raise ValueError("FlightDate contains missing or invalid dates")
    frame=frame.copy()
    frame["__month"]=dates.dt.to_period("M").astype(str)
    written=[]
    for month,group in frame.groupby("__month",sort=True):
        year,mm=map(int,month.split("-"))
        month_frame=group.drop(columns="__month")
        target=root/f"year={year}"/f"month={mm:02d}"/"flights.parquet"
        written.append(write_month(month_frame,target))
    return written

def refresh_and_update(start: str = START_MONTH, end: str = END_MONTH, refresh_sources: bool = True, partition_root: Path | None = None):
    profile = build(start=start, end=end, refresh=refresh_sources)
    input_path = OUT / profile["output_parquet"]
    target_root = partition_root or (OUT / "data_prepared" / "partitions")
    written = update_partitions(input_path, target_root)
    print("Обновлено месячных партиций:", len(written))
    return profile, written


In [13]:
# Для полной загрузки актуальных источников, слияния и обновления месячных разделов:
# refreshed_profile, updated_paths = refresh_and_update(
#     start=START_MONTH, end=END_MONTH, refresh_sources=True
# )
print("Обновление реализовано в refresh_and_update(); вызов оставлен ручным, чтобы обычный просмотр ноутбука не скачивал годовые архивы повторно.")

Обновление реализовано в refresh_and_update(); вызов оставлен ручным, чтобы обычный просмотр ноутбука не скачивал годовые архивы повторно.
